In [ ]:
from gp import GaussianProcess
import pandas as pd
import jax
import jax.numpy as jnp
import jax.random as jr

from sklearn.preprocessing import MinMaxScaler

In [ ]:
sequoia = pd.read_csv('../data/sequoia.csv')
sequoia.head()

In [ ]:
sequoia_nonan = sequoia[sequoia.signal_measured]

scaler = MinMaxScaler()
X_train = pd.DataFrame()
X_train[['latitude', 'longitude']] = scaler.fit_transform(sequoia_nonan[['latitude', 'longitude']])
X_train['indoor'] = sequoia_nonan['indoor'].astype(float).values

y_train = sequoia_nonan['rssi']

X_train = jnp.array(X_train)
y_train = jnp.array(y_train)

In [ ]:
indoor_mask = X_train[:,2].astype(bool)
outdoor_mask = ~X_train[:,2].astype(bool)

mean_in = y_train[indoor_mask].mean()
mean_out = y_train[outdoor_mask].mean()

def m(obs):
    return mean_in * obs[2] + mean_out * (1-obs[2])
    
def dist(obs1, obs2):
    '''
    Location distance only
    '''
    return jnp.linalg.norm(obs1[:2] - obs2[:2], ord=2)**2


# def dist(obs1, obs2):
#     '''
#     Location + indoor/outdoor
#     '''
#     return jnp.linalg.norm(obs1 - obs2, ord=2)**2


# def dist(obs1, obs2):
#     '''
#     Location + weighted indoor/outdoor
#     '''
#     loc_dist = jnp.linalg.norm(obs1[:2] - obs2[:2], ord=2)**2
#     inout_dist = jnp.abs(obs1[2] - obs2[2])
#     return loc_dist + (loc_dist**0.25  / jnp.sqrt(2)) * inout_dist

scale = min([y_train[indoor_mask].var(), y_train[outdoor_mask].var()])
bandwidth = 1e-2
def K(obs1, obs2):
    d = dist(obs1, obs2)
    return scale * jnp.exp(-d / (2*bandwidth))

In [ ]:
GP = GaussianProcess(m, K)

In [ ]:
GP.fit(X_train, y_train, Sigma_0=jnp.eye(X_train.shape[0])*y_train.var())

In [ ]:
grid_width = 100

x_new = jnp.linspace(0, 1, grid_width)
y_new = jnp.linspace(0, 1, grid_width)
grid_points = jnp.stack(jnp.meshgrid(x_new, y_new), axis=-1).reshape(-1, 2)
coord_grid = scaler.inverse_transform(grid_points)
long_new, lat_new = coord_grid.T

# match indoor to flag of closest observed point
from scipy.spatial import cKDTree
tree = cKDTree(sequoia[["latitude", "longitude"]].to_numpy())
_, indices = tree.query(coord_grid)
z_new = sequoia["indoor"].to_numpy()[indices].reshape(-1,1)


# z_new = jnp.ones([grid_width**2, 1]) * 0

X_new = jnp.concatenate([grid_points, z_new], axis=-1)

from wifiplotting import plot_wifi_heatmap, lonlat_to_world
df_new = pd.DataFrame(X_new, columns=['latitude', 'longitude', 'indoor'])
plot_wifi_heatmap(df_new, new_data=y_mean, aggregate=False)

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
candidates=jnp.array([0, 1]).reshape(-1,1)
X_best, y_mean, y_cov = GP.predict(X_new, candidates=None)

fig = plt.figure(figsize=(8, 6))
plt.scatter(lat_new, long_new, c=y_mean, cmap="RdYlGn")
plt.colorbar()
plt.ticklabel_format(style='plain', axis='both', useOffset=False)
plt.tight_layout()

In [ ]:
# GP.fit(X_train, y_train, Sigma_0=jnp.eye(X_train.shape[0])*1)
key = jr.PRNGKey(305)
for i in range(100):
    key = GP.gibbs(jr.fold_in(key, i))

In [ ]:
key = GP.gibbs(key)
print(GP.IG_posteriors)
print(GP.f_hat)

In [ ]:
f_hat = GP.f_hat
print(f"f_hat RMSE: {jnp.sqrt(jnp.mean((y_train - f_hat)**2)):.4f}")
print(f"Maximum RSSI Coordinates: ({long_new[f_hat.argmax()]:.5f}, {lat_new[f_hat.argmax()]:.5f}) ")

In [ ]:
from wifiplotting import plot_wifi_heatmap

In [ ]:
fig, ax, rssi_points, na_points = plot_wifi_heatmap(sequoia, show_na=0.5)
rows_used = int(rssi_points["sample_count"].sum() + na_points["sample_count"].sum())
print(f"RSSI heatmap uses {len(rssi_points) + len(na_points)} unique coordinate bins from {rows_used} rows.")

# doesn't work
# xs, ys = zip(*[lonlat_to_world(lon, lat, zoom) for lon, lat in zip(long_new, lat_new)])
# ax.scatter(xs, ys, c=y_mean, cmap="RdYlGn")

plt.show()